<a href="https://colab.research.google.com/github/vitor-dornela/FaesaBI/blob/main/C3/ETL/despesas_vitoria_2016.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# fonte dos dados https://transparencia.vitoria.es.gov.br/DadosAbertos.Lista.aspx

In [1]:
# Download dos arquivos - compatível com Windows e Colab
import requests
import os

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

downloads = [
    # O arquivo XLSX de 2016 esta retornando Erro 500 na API da Prefeitura, entao vamos usar apenas o CSV
    (
        'https://wstransparencia.vitoria.es.gov.br/api/despesas/csv?exercicio=2016&tipoDespesa=P',
        'despesas_vitoria_2016_P.csv'
    ),
]

for url, filename in downloads:
    print(f'Baixando {filename}...')
    response = requests.get(url, headers=HEADERS, stream=True)
    response.raise_for_status()
    with open(filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    size_kb = os.path.getsize(filename) / 1024
    print(f'  Salvo: {filename} ({size_kb:.1f} KB)\n')

C:\Users\vdmas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Baixando despesas_vitoria_2016_P.csv...
  Salvo: despesas_vitoria_2016_P.csv (18436.7 KB)



In [2]:
# Inspecionar CSV - compatível com Windows e Colab
import os

csv_file = 'despesas_vitoria_2016_P.csv'

# Tamanho do arquivo (substitui: file -i *.csv)
size_bytes = os.path.getsize(csv_file)
print(f'{csv_file}: tamanho={size_bytes} bytes, encoding=LATIN1 (inferido)')

# Primeiras 3 linhas (substitui: head -n 3 *.csv)
print()
with open(csv_file, 'r', encoding='latin1') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line, end='')

despesas_vitoria_2016_P.csv: tamanho=18879222 bytes, encoding=LATIN1 (inferido)

Ano,ValorOrcado,ValorAutorizado,ValorReservado,ValorEmpenhado,ValorLiquidado,ValorPago,CodUnidadeOrcamentaria,UnidadeOrcamentaria,CodUnidadeGestora,UnidadeGestora,CodOrgao,Orgao,CodFuncao,Funcao,CodSubFuncao,SubFuncao,CodPrograma,Programa,CodAcao,Acao,CodFonte,Fonte,CpfCnpjFornecedor,NumeroProcesso,NumeroContrato,CodNaturezaDespesa,NaturezaDespesa,NomeFornecedor,Data,IdEmpenho,Licitacao,Descricao,NumeroEmpenho
2016,,,,"473,00","473,00","473,00",,,228               ,SECRETARIA DE HABITACAO,280000,SECRETARIA DE HABITAÇÃO,16,Habitação                                         ,482,Habitação Urbana                                  ,0014,Programa Terra                                    ,1155,Aluguel Provisório                                ,1.000.0000,Recursos do Tesouro  Exercício Corrente,****76.377**,83401/2015,Não informado,33904899 ,Outros auxilios a pessoas fisicas                 ,ADALGETES MIRANDA DE C

In [ ]:
import pandas as pd
import warnings
import re

print("Lendo o arquivo bruto com Pandas...")
w_list = []
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    df_bruto = pd.read_csv('despesas_vitoria_2016_P.csv', engine='python', on_bad_lines='warn', encoding='latin1')
    w_list = [str(warn.message) for warn in w if issubclass(warn.category, pd.errors.ParserWarning)]

linhas_afetadas = []
for msg in w_list:
    match = re.search(r'line (\d+)', msg)
    if match:
        linhas_afetadas.append(int(match.group(1)))

print(f"Total de warnings (bad_lines): {len(linhas_afetadas)}")
if linhas_afetadas:
    print(f"Linhas afetadas (amostra de 5): {linhas_afetadas[:5]}")
    
    print("\n--- AMOSTRA DA PRIMEIRA LINHA COM PROBLEMA NO ARQUIVO BRUTO ---")
    with open('despesas_vitoria_2016_P.csv', 'r', encoding='latin1') as f:
        for i, line in enumerate(f, start=1):
            if i == linhas_afetadas[0]:
                print(f"Linha {i}:")
                print(line)
                break


In [ ]:
import csv
import io
import pandas as pd

def read_cleaned_csv(filepath):
    cleaned_rows = []
    
    with open(filepath, 'r', encoding='latin1') as f:
        reader = csv.reader(f)
        header = next(reader)
        current_row = []
        
        for row in reader:
            if not current_row:
                if not row: continue
                current_row = row
            else:
                if not row:
                    current_row[-1] += '\n'
                else:
                    current_row[-1] = current_row[-1] + '\n' + row[0]
                    current_row.extend(row[1:])
            
            if len(current_row) < 34:
                continue
                
            if len(current_row) > 34:
                excess = len(current_row) - 34
                desc_pieces = current_row[32:33+excess]
                desc = ",".join(desc_pieces).replace('"', "'")
                current_row = current_row[:32] + [desc] + [current_row[-1]]
                
            cleaned_rows.append(current_row)
            current_row = []
            
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(header)
    writer.writerows(cleaned_rows)
    output.seek(0)
    return output

print("Executando a limpeza das aspas quebradas...")
cleaned_csv_io = read_cleaned_csv('despesas_vitoria_2016_P.csv')
print("Limpeza concluída! Dados estruturados na memória.")


In [ ]:
import pandas as pd
import warnings

print("Lendo os dados limpos com Pandas...")
w_list = []
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    cleaned_csv_io.seek(0)
    dfcsv = pd.read_csv(cleaned_csv_io, engine='python', on_bad_lines='warn')
    w_list = [str(warn.message) for warn in w if issubclass(warn.category, pd.errors.ParserWarning)]

print(f"Total de warnings (bad_lines) APÓS limpeza: {len(w_list)}")

if len(w_list) > 0:
    print("Amostra dos warnings encontrados:")
    for warn in w_list[:3]:
        print(f" - {warn}")
else:
    print("✅ Excelente! Nenhuma linha quebrou o Parser. Os dados estão perfeitos estruturalmente.")
    
# Mostra uma amostra do dataframe limpo
dfcsv.head(3)
